<a href="https://colab.research.google.com/github/mancinigabriel/tcc-pece-assin2-llm-challenges/blob/main/notebooks/Ministral3_Reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Preparando ambiente

In [1]:
!git clone https://github.com/mancinigabriel/tcc-pece-assin2-llm-challenges.git

Cloning into 'tcc-pece-assin2-llm-challenges'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 95 (delta 45), reused 63 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 113.99 KiB | 9.50 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [2]:
!pip install transformers==5.0.0rc0
!pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 39.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 135.1 MB/s eta 0:00:00


In [3]:
import sys
import os

PROJECT_ROOT = os.path.abspath('/content/tcc-pece-assin2-llm-challenges')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend, FineGrainedFP8Config
from datetime import datetime, timezone, timedelta
from src import prompts
from src import utils
from src import data
from src import metrics
import pandas as pd
import random
import torch
import re
import time

In [6]:
cfg = utils.load_config(
    "/content/tcc-pece-assin2-llm-challenges/configs/base.yaml",
    "/content/tcc-pece-assin2-llm-challenges/configs/models/mistral3.yaml"
)

generation_args = cfg["generation"]

gpu = 'A100 RAM alta'

#Ministral 3 - 3B - Reasoning

In [7]:
timing = {}

timing['inicio'] = utils.time_log()


model_id = "mistralai/Ministral-3-3B-Reasoning-2512"
model_name = "mistral3b_reasoning"
quantizado = False
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

df_assin_2 = data.gera_df()

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

##Teste de Consistência

In [8]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:07:46
100 - 2026-01-12 02:07:56
200 - 2026-01-12 02:08:07
300 - 2026-01-12 02:08:17
400 - 2026-01-12 02:08:28


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:08:39
100 - 2026-01-12 02:08:49
200 - 2026-01-12 02:09:00
300 - 2026-01-12 02:09:10
400 - 2026-01-12 02:09:21


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


...

/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:10:23
100 - 2026-01-12 02:10:33
200 - 2026-01-12 02:10:44
300 - 2026-01-12 02:10:54
400 - 2026-01-12 02:11:04


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-12 02:11:15
100 - 2026-01-12 02:11:25
200 - 2026-01-12 02:11:36
300 - 2026-01-12 02:11:46
400 - 2026-01-12 02:11:56


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))


##Loop de aplicação do prompt em todo o dataset

In [ ]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

0 - 2026-01-12 02:12:11
100 - 2026-01-12 02:12:22
200 - 2026-01-12 02:12:32
300 - 2026-01-12 02:12:43
400 - 2026-01-12 02:12:53
500 - 2026-01-12 02:13:03
600 - 2026-01-12 02:13:14
700 - 2026-01-12 02:13:24
800 - 2026-01-12 02:13:35
900 - 2026-01-12 02:13:45
1000 - 2026-01-12 02:13:56
1100 - 2026-01-12 02:14:06
1200 - 2026-01-12 02:14:17
1300 - 2026-01-12 02:14:27
1400 - 2026-01-12 02:14:37
1500 - 2026-01-12 02:14:48
1600 - 2026-01-12 02:14:58
1700 - 2026-01-12 02:15:09
1800 - 2026-01-12 02:15:19
1900 - 2026-01-12 02:15:30
2000 - 2026-01-12 02:15:40
2100 - 2026-01-12 02:15:50
2200 - 2026-01-12 02:16:01
2300 - 2026-01-12 02:16:11
2400 - 2026-01-12 02:16:22
2500 - 2026-01-12 02:16:32
2600 - 2026-01-12 02:16:43
2700 - 2026-01-12 02:16:53
2800 - 2026-01-12 02:17:03
2900 - 2026-01-12 02:17:14
3000 - 2026-01-12 02:17:24
3100 - 2026-01-12 02:17:35
3200 - 2026-01-12 02:17:45
3300 - 2026-01-12 02:17:55
3400 - 2026-01-12 02:18:06
3500 - 2026-01-12 02:18:16
3600 - 2026-01-12 02:18:27
3700 - 2026-0

# Mistral 3 - 8B - Reasoning

In [ ]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = "mistralai/Ministral-3-8B-Reasoning-2512"
model_name = "mistral8b_reasoning"
quantizado = False
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

df_assin_2 = data.gera_df()

In [ ]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

In [ ]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)